# ESM-2 Scaling with Real Protein Sequences (UniRef)

Tests the hypothesis with **ecological validity**: Do real protein sequences show the same geometric instability trend?

## Key Difference from Synthetic Experiment

- **Synthetic**: Random amino acid sequences -- tests pure architectural property
- **UniRef (this notebook)**: Real protein sequences -- tests if the effect holds in practice

Uses **UniRef50** database (clustered at 50% identity) for diverse, real-world proteins.

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (Runtime > Change runtime type > GPU)
3. Run cells in order

---

In [ ]:
# Install Dependencies
print("Installing dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas
!pip install -q datasets  # HuggingFace datasets (primary source)
!pip install -q biopython  # Fallback for parsing UniRef FASTA
print("Done!")

In [ ]:
# Configuration

import sys, os
sys.path.insert(0, '.')
PHASE = 'full'  # Start here!

OUTPUT_BASE = './results/esm2_uniref/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    # Quick validation with real proteins
    'quick': {
        'n_sequences': 500,
        'min_length': 100,
        'max_length': 400,
        'models': [
            'facebook/esm2_t6_8M_UR50D',
            'facebook/esm2_t33_650M_UR50D',
        ],
        'batch_size': 16,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    # Full experiment with real proteins
    'full': {
        'n_sequences': 10000,
        'min_length': 100,
        'max_length': 400,
        'models': [
            'facebook/esm2_t6_8M_UR50D',
            'facebook/esm2_t12_35M_UR50D',
            'facebook/esm2_t30_150M_UR50D',
            'facebook/esm2_t33_650M_UR50D',
            'facebook/esm2_t36_3B_UR50D',
            'facebook/esm2_t48_15B_UR50D',
        ],
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Data source: UniRef50 (real proteins)")
print(f"Sequences: {config['n_sequences']}")
print(f"Length range: {config['min_length']}-{config['max_length']} amino acids")
print(f"Models: {len(config['models'])}")

In [ ]:
# Load Real Protein Sequences
#
# Strategy: Try HuggingFace datasets first (fast, reliable),
# then fall back to streaming UniRef50 FASTA, then synthetic.

import os
import numpy as np

VALID_AAS = set('ACDEFGHIKLMNPQRSTVWY')
MAX_SCAN_RECORDS = 200_000  # Cap scanning to avoid long downloads


def load_from_huggingface(n_sequences, min_len, max_len, seed=320):
    """
    Load protein sequences from a HuggingFace UniRef dataset.
    Much faster and more reliable than streaming the 40GB FASTA.
    """
    try:
        from datasets import load_dataset
        print("Loading UniRef50 from HuggingFace datasets...")

        # Use a pre-processed UniRef50 subset
        ds = load_dataset(
            'sagawa/uniref50-sample',
            split='train',
            streaming=True,
        )

        rng = np.random.default_rng(seed)
        sequences = []
        scanned = 0

        for record in ds:
            seq = record.get('sequence', record.get('text', ''))
            if not seq:
                continue
            scanned += 1

            # Filter by length and valid amino acids
            if min_len <= len(seq) <= max_len:
                if all(c in VALID_AAS for c in seq):
                    sequences.append(seq)

            if scanned % 10000 == 0:
                print(f"  Scanned {scanned}, collected {len(sequences)}/{n_sequences}", end='\r')

            if len(sequences) >= n_sequences:
                break
            if scanned >= MAX_SCAN_RECORDS:
                break

        print(f"\nCollected {len(sequences)} sequences from HuggingFace")
        return sequences if len(sequences) >= n_sequences * 0.8 else None

    except Exception as e:
        print(f"HuggingFace loading failed: {e}")
        return None


def load_from_fasta_stream(n_sequences, min_len, max_len, seed=320):
    """
    Stream UniRef50 FASTA with reservoir sampling.
    Capped at MAX_SCAN_RECORDS to avoid timeout.
    """
    import urllib.request
    import gzip
    from Bio import SeqIO

    UNIREF_URL = 'https://ftp.uniprot.org/pub/databases/uniprot/uniref/uniref50/uniref50.fasta.gz'

    print(f"Streaming UniRef50 FASTA (scanning up to {MAX_SCAN_RECORDS:,} records)...")
    rng = np.random.default_rng(seed)
    sequences = []
    eligible_count = 0

    try:
        with urllib.request.urlopen(UNIREF_URL, timeout=60) as response:
            with gzip.GzipFile(fileobj=response) as f:
                for i, record in enumerate(SeqIO.parse(f, 'fasta')):
                    seq = str(record.seq)

                    if min_len <= len(seq) <= max_len:
                        if all(c in VALID_AAS for c in seq):
                            eligible_count += 1
                            # Reservoir sampling
                            if len(sequences) < n_sequences:
                                sequences.append(seq)
                            else:
                                j = rng.integers(0, eligible_count)
                                if j < n_sequences:
                                    sequences[j] = seq

                    if (i + 1) % 10000 == 0:
                        print(f"  Scanned {i+1:,}, collected {len(sequences)}/{n_sequences}", end='\r')

                    # Cap total scanning
                    if i + 1 >= MAX_SCAN_RECORDS:
                        break

        print(f"\nCollected {len(sequences)} sequences from FASTA stream")
        return sequences if len(sequences) >= n_sequences * 0.8 else None

    except Exception as e:
        print(f"FASTA streaming failed: {e}")
        return None


def generate_synthetic_fallback(n_sequences, min_len, max_len, seed=320):
    """Last resort: random amino acid sequences."""
    print("WARNING: Using synthetic sequences as fallback")
    AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')
    rng = np.random.default_rng(seed)
    avg_len = (min_len + max_len) // 2
    return [
        ''.join(rng.choice(AMINO_ACIDS, size=avg_len))
        for _ in range(n_sequences)
    ]


def load_sequences(config, seed=320):
    """Load sequences with cascading fallback."""
    n = config['n_sequences']
    lo, hi = config['min_length'], config['max_length']

    cache_file = f'{CACHE_DIR}/uniref_sample_{n}.txt'
    if os.path.exists(cache_file):
        print(f"Loading cached UniRef sequences from {cache_file}...")
        with open(cache_file) as f:
            sequences = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(sequences)} cached sequences")
        return sequences

    # Try HuggingFace first, then FASTA, then synthetic
    sequences = load_from_huggingface(n, lo, hi, seed)
    if sequences is None:
        sequences = load_from_fasta_stream(n, lo, hi, seed)
    if sequences is None:
        sequences = generate_synthetic_fallback(n, lo, hi, seed)

    # Truncate to exact count
    sequences = sequences[:n]

    # Cache
    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(cache_file, 'w') as f:
        f.write('\n'.join(sequences))
    print(f"Cached to {cache_file}")

    return sequences


# Load sequences
sequences = load_sequences(config, seed=320)

print(f"\nSequence stats:")
print(f"  Count: {len(sequences)}")
print(f"  Length range: {min(len(s) for s in sequences)}-{max(len(s) for s in sequences)}")
print(f"  Mean length: {np.mean([len(s) for s in sequences]):.0f}")
print(f"  Sample: {sequences[0][:50]}...")

In [ ]:
# Protein Perturbation Suite

from dataclasses import dataclass, field

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_protein(sequence, mutation_rate, rng):
    """Randomly substitute amino acids."""
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [aa for aa in AMINO_ACIDS if aa != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_sequence(sequence):
    return sequence[::-1]


class ProteinPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_reverse=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_reverse = include_reverse

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"aa_sub_{int(rate * 100)}pct"
            perturbed = [mutate_protein(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='substitution', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'AA substitutions at {rate*100:.0f}% rate',
            )
        if self.include_reverse:
            results['reverse'] = PerturbedSet(
                name='reverse', category='reverse',
                sequences=[reverse_sequence(s) for s in sequences],
                params={}, description='Sequence reversal',
            )
        return results


suite = ProteinPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:5], pset.sequences)
    ]
    print(f"{name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# ESM-2 Model Wrapper

import torch
import gc
from transformers import AutoTokenizer, AutoModel


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def make_esm2_embedding_fn(model_name, batch_size=8):
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Device: {device} | Params: {n_params:.1f}M")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1
            if batch_idx % 10 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')
            tokens = tokenizer(batch, return_tensors='pt', padding=True,
                             truncation=True, max_length=1024)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            outputs = model(**tokens)
            hidden = outputs.last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings.append(pooled.cpu().numpy())
        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer

print("Model wrapper ready")

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os
import time
import pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'ESM-2 + UNIREF50 EXPERIMENT -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
all_detailed_rows = []  # Per-perturbation detailed results

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()

    embed_fn, model_obj, tokenizer_obj = make_esm2_embedding_fn(
        model_name, config['batch_size']
    )

    # Get actual param count for detailed CSV
    n_params_m = sum(p.numel() for p in model_obj.parameters()) / 1e6

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = model_name.replace('/', '_') + '_uniref'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')

    print(f'  Shape: {embeddings_clean.shape}')

    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    del model_obj, tokenizer_obj
    cleanup_gpu()

    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    # Collect per-perturbation rows
    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name,
            'size_M': round(n_params_m),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:     {summary["mean_perturbation_stability_score"]:.4f}')
    print(f'  Pert. Magnitude:     {summary["mean_perturbation_magnitude"]:.4f}')

# Save detailed per-perturbation CSV
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/esm2_uniref_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Results & Visualization

from evaluation_harness import compare_models
import matplotlib.pyplot as plt

comparison = compare_models(reports, output_dir=RESULTS_DIR)

print('\n' + '=' * 70)
print('ESM-2 + UNIREF50 SUMMARY')
print('=' * 70 + '\n')

for model_name, data in comparison['models'].items():
    s = data['summary']
    short = model_name.split('/')[-1]
    print(f'{short}:')
    print(f'  Composite Stability:  {s["mean_composite_stability"]:.4f} +/- {s["std_composite_stability"]:.4f}')
    print()

# Scaling plot
summaries = [r.summary() for r in reports]
model_names = [r.model_name for r in reports]

SIZE_MAP = {'8M': 8, '35M': 35, '150M': 150, '650M': 650, '3B': 3000}
sizes = [SIZE_MAP.get(k, 0) for n in model_names for k in SIZE_MAP if k in n]
values = [s['mean_composite_stability'] for s in summaries]

plt.figure(figsize=(10, 6))
plt.plot(sizes, values, 'o-', linewidth=2, markersize=10)
plt.xscale('log')
plt.xlabel('Model Size (millions of parameters)')
plt.ylabel('Composite Stability')
plt.title('ESM-2 on Real Proteins (UniRef50): Stability vs. Size', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.savefig(f'{RESULTS_DIR}/esm2_uniref_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved to results/esm2_uniref_scaling_{PHASE}.png')

In [ ]:
# Cross-Experiment Comparison
#
# Overlay UniRef results with synthetic ESM-2 results (if available)
# to confirm the geometric tax holds on real proteins.

import matplotlib.pyplot as plt
import pandas as pd
import glob

# Try to load synthetic ESM-2 results
synthetic_files = glob.glob(f'{RESULTS_DIR}/esm2_scaling_*_detailed.csv')
if not synthetic_files:
    # Also check parent/sibling directories
    synthetic_files = glob.glob('../**/esm2_scaling_*_detailed.csv', recursive=True)

if synthetic_files:
    print(f'Found synthetic results: {synthetic_files[0]}')
    df_synthetic = pd.read_csv(synthetic_files[0])

    # Aggregate by model
    syn_agg = df_synthetic.groupby(['model', 'size_M'])['composite'].mean().reset_index()
    uni_agg = df_detailed.groupby(['model', 'size_M'])['composite'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(syn_agg['size_M'], syn_agg['composite'], 'o--',
            color='tab:gray', linewidth=2, markersize=10, label='ESM-2 (Synthetic)', alpha=0.6)
    ax.plot(uni_agg['size_M'], uni_agg['composite'], 's-',
            color='tab:blue', linewidth=2, markersize=10, label='ESM-2 (UniRef50)')

    ax.set_xscale('log')
    ax.set_xlabel('Model Size (millions of parameters)', fontsize=12)
    ax.set_ylabel('Composite Stability', fontsize=12)
    ax.set_title('Synthetic vs. Real Proteins: Geometric Tax Validation', fontweight='bold', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/esm2_synthetic_vs_uniref_{PHASE}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved comparison plot')
else:
    print('No synthetic ESM-2 results found.')
    print('Run the synthetic experiment first, or copy its results CSV to results/')
    print('Expected filename pattern: results/esm2_scaling_*_detailed.csv')